# OE1: comprobación reproducible en Google Colab

**Objetivo oficial (literal).** Determinar la relación entre el cumplimiento de los requerimientos funcionales y no funcionales de la arquitectura multiagente y el desempeño del proceso de adaptación de contenido multimodal, mediante indicadores cuantificables de funcionalidad, confiabilidad, eficiencia y calidad de adaptación.

**Qué comprueba este cuaderno.** (a) Recalcula, desde cada petición registrada por el generador de carga, los indicadores por corrida (peticiones, errores, ventana, throughput, P50, media y P95 de la latencia, T_conv medio) de las 120 corridas válidas y los compara con lo guardado por el ejecutor; (b) reejecuta el análisis pre-registrado completo (puntaje de cumplimiento S, requerimientos, confiabilidad, eficiencia, calidad de adaptación, contrastes pareados, correlaciones intra-bloque e interferencia) y lo compara con el resultado local.

**Datos de entrada.** Registros por petición `raw_http/*.csv` y banderas por ciclo `*.oe1.csv` de la campaña, `runs.jsonl`, `intentos.jsonl`, perfiles del conjunto de datos, referencia gold sellada, monitor de recursos y el script de análisis sellado (`analisis_oe1.py`), con los resultados locales para comparar.

**Qué no hace.** No recalcula los tiempos medidos, que son los del equipo local con interferencia registrada. No repite las campañas ni regenera datos, no modifica archivos originales, scripts ni umbrales, y no convierte ninguna campaña en controlada. Ejecute las celdas en orden en un entorno nuevo de Colab; la carga de datos pide subir los archivos indicados.

**Contenido (diez pasos).** 1 descripción; 2 datos de entrada; 3 verificación de archivos y estructura; 4 dependencias; 5 ejecución del análisis; 6 reconstrucción de los indicadores; 7 comparación con los resultados oficiales; 8 diferencias numéricas; 9 verificación de las salidas finales; 10 conclusión de reproducibilidad (calculada por la última celda).

In [ ]:
import sys, platform, os
print('Python', sys.version); print(platform.platform())
!pip -q install numpy scipy pandas statsmodels
import numpy, scipy, pandas, statsmodels; print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| pandas', pandas.__version__, '| statsmodels', statsmodels.__version__)
print('CPU', os.cpu_count())

In [ ]:
import zipfile, glob
PARTES = 1
NECESARIOS = []
partes = sorted(glob.glob('oe1_http_p*.zip'))
faltan = [z for z in NECESARIOS if not os.path.exists(z)]
if len(partes) < PARTES or faltan:
    from google.colab import files
    print('Suba los archivos:', NECESARIOS, 'y las', PARTES, 'partes oe1_http_p1.zip ... oe1_http_p'+str(PARTES)+'.zip')
    files.upload()
    partes = sorted(glob.glob('oe1_http_p*.zip'))
assert len(partes) == PARTES, ('partes encontradas:', partes)
assert all(os.path.exists(z) for z in NECESARIOS), 'faltan paquetes de analisis'
for z in NECESARIOS + partes: zipfile.ZipFile(z).extractall('.')
print(len(partes), 'partes http extraidas + paquetes de analisis', NECESARIOS, '|', sorted(os.listdir('http')))

In [ ]:
import hashlib
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
bad=[]; n=0
for l in open('http/hashes_paquete_http_oe1.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); n+=1
    if sha(os.path.join('http',nm))!=h: bad.append(nm)
print('archivos del paquete HTTP verificados contra el hash local:', n, '| distintos:', bad[:5]); assert not bad
base='http/oe1/campana_01'; ok=0; mal=[]
for l in open(base+'/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join(base,nm)
    if os.path.exists(p):
        ok+=1
        if sha(p)!=h: mal.append(nm)
print('oe1: archivos de la campaña presentes y verificados contra su SHA256SUMS.txt:', ok, '| distintos:', mal); assert not mal

## Recálculo por corrida desde las peticiones registradas

In [ ]:
os.makedirs('http/out', exist_ok=True)
for k in ('oe1',):
    !python http/recomputo_http.py http/{k}/campana_01 http/out/recomputo_{k}.csv http/out/resumen_{k}.json > http/out/log_{k}.txt 2>&1
import json
print('Recálculo INDEPENDIENTE, desde cada petición registrada por el generador de carga, de los indicadores por corrida (12 campos) y comparación con lo guardado por el ejecutor:')
for k in ('oe1',):
    r=json.load(open(f'http/out/resumen_{k}.json'))
    mx=max(r['diferencia_abs_max_por_campo'].values()); mr=max(r['diferencia_rel_max_por_campo'].values())
    print(f"{k}: intentos recalculados {r['intentos_con_raw_recalculados']} | corridas válidas finales recalculadas {r['corridas_validas_finales_recalculadas']} | con los 12 campos idénticos {r['corridas_validas_finales_con_todos_los_campos_iguales']} | diferencia absoluta máxima {mx} | relativa máxima {mr}")

## Análisis pre-registrado de OE1 desde los datos originales

In [ ]:
!python http/oe1/analisis_oe1.py http/oe1/campana_01 http/oe1/salida http/oe1/gold_sellado.json http/oe1/campana_01_monitor.csv > http/oe1/log_analisis.txt 2>&1
print(open('http/oe1/log_analisis.txt').read()[-300:])
for f in ('celdas.csv','corridas_indicadores.csv','efectos.csv','resultados_oe1.json'):
    print(f, 'SHA-256 idéntico al resultado local:', sha('http/oe1/salida/'+f)==sha('http/oe1/esperado_local/'+f))

In [ ]:
import pandas as pd
import json, math
def hoja(x, p=''):
    if isinstance(x, dict):
        for k,v in x.items(): yield from hoja(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i,v in enumerate(x): yield from hoja(v, p+'/'+str(i))
    else: yield p, x
def comparar_json(ruta_colab, ruta_local, etiqueta):
    A=dict(hoja(json.load(open(ruta_colab,encoding='utf8')))); B=dict(hoja(json.load(open(ruta_local,encoding='utf8'))))
    mismas = set(A)==set(B); mx=0.0; mxrel=0.0; peor=None; n_num=0; n_dist=0; n_nonum=0; dist=[]
    for k in A:
        if k not in B: continue
        va, vb = A[k], B[k]
        if isinstance(va,(int,float)) and isinstance(vb,(int,float)) and not isinstance(va,bool) and not isinstance(vb,bool):
            n_num+=1; dif=abs(va-vb); rel=dif/max(abs(vb),1e-300) if dif>0 else 0.0
            if dif>mx: mx=dif; peor=(k,va,vb)
            mxrel=max(mxrel, rel)
            if dif>0: n_dist+=1
            if dif>1e-9 and rel>1e-9: dist.append((k,va,vb,dif,rel))
        else:
            n_nonum+=1
            if va!=vb: dist.append((k,va,vb,None,None))
    print(f'[{etiqueta}] mismas claves: {mismas} | hojas: {len(A)} (numericas {n_num}, no numericas {n_nonum})')
    print(f'   diferencia absoluta maxima: {mx:.3e} | relativa maxima: {mxrel:.3e} | hojas numericas con diferencia > 0: {n_dist} | hojas con diferencia > 1e-9 (abs y rel): {len(dist)}')
    print('   peor hoja:', peor)
    for d_ in dist[:20]: print('   DIFERENCIA', d_)
def comparar_csv(ruta_colab, ruta_local, etiqueta):
    a=pd.read_csv(ruta_colab); b=pd.read_csv(ruta_local)
    ok = list(a.columns)==list(b.columns) and a.shape==b.shape
    mx=0.0; mxrel=0.0; nd=0
    if ok:
        for c in a.columns:
            if pd.api.types.is_numeric_dtype(a[c]) and pd.api.types.is_numeric_dtype(b[c]) and not pd.api.types.is_bool_dtype(a[c]) and not pd.api.types.is_bool_dtype(b[c]):
                dif=(a[c]-b[c]).abs().fillna(0); mx=max(mx,float(dif.max()))
                rel=(dif/b[c].abs().clip(lower=1e-300)).where(dif>0,0.0); mxrel=max(mxrel,float(rel.max())); nd+=int((dif>0).sum())
            else:
                nd+=int((a[c].astype(object).where(a[c].notna(),'<NA>').astype(str)!=b[c].astype(object).where(b[c].notna(),'<NA>').astype(str)).sum())
    print(f'[{etiqueta}] misma forma y columnas: {ok} {a.shape} | dif abs max: {mx:.3e} | rel max: {mxrel:.3e} | celdas distintas: {nd}')
comparar_json('http/oe1/salida/resultados_oe1.json','http/oe1/esperado_local/resultados_oe1.json','OE1 pre-registrado: resultados_oe1.json')
for f in ('celdas.csv','corridas_indicadores.csv','efectos.csv'):
    comparar_csv('http/oe1/salida/'+f,'http/oe1/esperado_local/'+f,'OE1 pre-registrado: '+f)

In [ ]:
a=json.load(open('http/oe1/salida/resultados_oe1.json')); b=json.load(open('http/oe1/esperado_local/resultados_oe1.json'))
print('registros/válidas/interferidas:', [a['n_registros'],a['n_validas'],a['interferidas']], '|', [b['n_registros'],b['n_validas'],b['interferidas']])
print('variación de S (min, max, niveles):', a['variacion_de_S'], '|', b['variacion_de_S'])
for k in ('p95','thr','tconv','f1','c','err'):
    print('rho intra-bloque con S', k, round(a['relacion_S_respuesta'][k]['rho_intra_bloque'],4), '|', round(b['relacion_S_respuesta'][k]['rho_intra_bloque'],4))
for phi in ('0.0','0.02','0.05'):
    print('I-13 (Wilson, ciclos agrupados) phi', phi, round(a['confiabilidad_I13'][phi]['ciclos_pool']['est'],4), '|', round(b['confiabilidad_I13'][phi]['ciclos_pool']['est'],4))

## Conclusión de reproducibilidad

In [ ]:
import json, os
def _hojas(x, p=''):
    if isinstance(x, dict):
        for k, v in x.items(): yield from _hojas(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i, v in enumerate(x): yield from _hojas(v, p+'/'+str(i))
    else: yield p, x
def _verif(a, b):
    A = dict(_hojas(json.load(open(a, encoding='utf8')))); B = dict(_hojas(json.load(open(b, encoding='utf8'))))
    mx = 0.0; ng = 0; malas = []
    for k in A:
        if k in B and isinstance(A[k], (int, float)) and isinstance(B[k], (int, float)) and not isinstance(A[k], bool) and not isinstance(B[k], bool):
            d = abs(A[k]-B[k]); mx = max(mx, d)
            if d > 1e-9 and d/max(abs(B[k]), 1e-300) > 1e-9: ng += 1; malas.append((k, A[k], B[k]))
        elif k in B and A[k] != B[k]: ng += 1; malas.append((k, A[k], B[k]))
    return set(A) == set(B), len(A), mx, ng, malas
CHECKS = [('OE1 resultados_oe1.json', 'http/oe1/salida/resultados_oe1.json', 'http/oe1/esperado_local/resultados_oe1.json')]
HTTP = ['oe1']
print('CONCLUSION DE REPRODUCIBILIDAD')
todo_ok = True
dep_version = False
for etiqueta, a, b in CHECKS:
    mismas, n, mx, ng, malas = _verif(a, b)
    solo_wilcoxon = ng > 0 and all(k.endswith('/p_wilcoxon') for k, _, _ in malas)
    ok = mismas and ng == 0; todo_ok &= ok or (mismas and solo_wilcoxon); dep_version |= (not ok) and mismas and solo_wilcoxon
    veredicto = 'COINCIDE' if ok else ('COINCIDE salvo p_wilcoxon (aproximación normal que depende de la versión de scipy)' if mismas and solo_wilcoxon else 'NO COINCIDE')
    print(f'  {etiqueta}: misma estructura {mismas} | hojas {n} | diferencia absoluta máxima {mx:.3e} | hojas con diferencia > 1e-9: {ng} -> {veredicto}')
    for k, x, y in malas[:10]: print(f'     hoja distinta: {k} | Colab {x!r} | local {y!r}')
for k in HTTP:
    r = json.load(open(f'http/out/resumen_{k}.json'))
    ok = r['corridas_validas_finales_recalculadas'] == r['corridas_validas_finales_con_todos_los_campos_iguales'] and max(r['diferencia_abs_max_por_campo'].values()) == 0
    todo_ok &= ok
    print(f"  {k}: indicadores por corrida recalculados desde las peticiones: {r['corridas_validas_finales_recalculadas']} corridas válidas, {r['corridas_validas_finales_con_todos_los_campos_iguales']} idénticas -> {'COINCIDE' if ok else 'NO COINCIDE'}")
if todo_ok and dep_version: print('Resultado: el análisis se reproduce con los datos oficiales; todas las hojas coinciden dentro de 1e-9 salvo los p de Wilcoxon listados arriba, que dependen de la versión de scipy instalada (scipy', __import__('scipy').__version__, 'en este entorno). Las demás cantidades, incluido el ajuste de Holm calculado con p_t, coinciden.')
else: print('Resultado:', 'el análisis se reproduce con los datos oficiales y coincide con el resultado local dentro de la tolerancia de 1e-9 (las diferencias de coma flotante se informan arriba).' if todo_ok else 'HAY DIFERENCIAS: revise las líneas marcadas como NO COINCIDE antes de concluir.')
print('Alcance: se vuelve a ejecutar el análisis estadístico sobre los datos de entrada; no se repiten las campañas ni se regeneran los ciclos o las corridas, y la reproducción no implica que las condiciones de adquisición fueran controladas.')